<a href="https://colab.research.google.com/github/KaifAhmad1/AB-Testing/blob/master/Product_Marketing_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Sample Data for AI Product Marketing System for Image Generation
# Product Marketing AI System

**Overview:**

The Product Marketing AI System automatically creates high-quality marketing visuals by processing, enhancing, and refining input images.

**Key Steps:**
- **Input & Preprocessing:**  
  Users provide a base image (e.g., a background or scene) and a secondary image (e.g., a product or model). The system enhances these images, adjusts resolution and contrast, and generates segmentation masks.

- **Context & Prompt Generation:**  
  It analyzes the images to extract contextual details and automatically generates a detailed, structured marketing prompt tailored to the product and target audience.

- **Image Generation & Refinement:**  
  Advanced AI models use the prompt to generate photorealistic marketing images. The system then refines the results based on user feedback to ensure the output meets professional quality standards.

- **Quality Evaluation:**  
  Final images are evaluated using metrics (such as SSIM, PSNR, and color histograms) to compare them with the original inputs. Additionally, AI-driven feedback helps further improve image quality.

**Applications:**
- **Digital Advertising & E-Commerce:**  
  Generate stunning visuals for online stores, social media campaigns, and ad banners.
- **Branding & Marketing:**  
  Enhance and standardize visual branding materials across various industries such as fashion, tech, automotive, and real estate.
- **Content Creation:**  
  Streamline production of professional digital content for websites, promotions, and digital signage.

This system provides a seamless, automated solution for creating visually appealing marketing assets, reducing manual effort and ensuring consistency and high quality across all outputs.

In [1]:
import os
import requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

# Optional: Try to import Crawl4AI for better crawling; will use basic requests-based crawling if not available.
try:
    from crawl4ai import CrawlEngine
    crawl4ai_available = True
except ImportError:
    print("Crawl4AI library not installed. Falling back to basic requests-based crawling.")
    crawl4ai_available = False

# Base directory for downloaded images; separate subfolders for base and secondary images.
BASE_DIR = "crawl4ai_images"
base_images_folder = os.path.join(BASE_DIR, "base_images")
secondary_images_folder = os.path.join(BASE_DIR, "secondary_images")
os.makedirs(base_images_folder, exist_ok=True)
os.makedirs(secondary_images_folder, exist_ok=True)

def is_valid(url):
    parsed = urlparse(url)
    return bool(parsed.netloc) and bool(parsed.scheme)

def extract_images(html, base_url):
    """
    Extracts image URLs from the provided HTML content using BeautifulSoup.
    """
    soup = BeautifulSoup(html, "html.parser")
    image_urls = []
    for img in soup.find_all("img"):
        img_url = img.get("src")
        if not img_url:
            continue
        img_url = urljoin(base_url, img_url)
        if is_valid(img_url) and any(img_url.lower().endswith(ext) for ext in [".jpg", ".jpeg", ".png", ".gif"]):
            image_urls.append(img_url)
    return image_urls

def download_image(url, folder):
    """
    Downloads an image from the provided URL and saves it in the specified folder.
    """
    try:
        print(f"Downloading: {url}")
        response = requests.get(url, stream=True, timeout=10)
        response.raise_for_status()
        image_name = os.path.basename(url.split("?")[0])
        save_path = os.path.join(folder, image_name)
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
        print(f"Saved image to {save_path}")
    except Exception as e:
        print(f"Error downloading {url}: {e}")

def fetch_page(url):
    """
    Retrieves page content using Crawl4AI if available; otherwise uses a basic requests approach.
    """
    if crawl4ai_available:
        engine = CrawlEngine()
        print(f"Using Crawl4AI engine for URL: {url}")
        return engine.fetch_page(url)
    else:
        print(f"Falling back to basic requests for URL: {url}")
        try:
            resp = requests.get(url, timeout=10)
            resp.raise_for_status()
            return resp.text
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            return None

def crawl_and_download(url, folder, limit=5):
    """
    Crawls the given URL, extracts image links, and downloads up to 'limit' images.
    """
    print(f"\nCrawling: {url}")
    content = fetch_page(url)
    if not content:
        print(f"Skipping {url} due to fetch error.")
        return
    image_urls = extract_images(content, url)
    print(f"Found {len(image_urls)} images on {url}")
    for idx, img_url in enumerate(image_urls[:limit]):
        download_image(img_url, folder)

if __name__ == "__main__":
    # Updated source URLs for base images (more diverse backgrounds)
    base_sources = [
        "https://unsplash.com/s/photos/office",
        "https://www.pexels.com/search/urban/",
        "https://pixabay.com/images/search/landscape/",
        "https://www.flickr.com/search/?text=studio",
        "https://stock.adobe.com/search?k=professional+background",
        "https://unsplash.com/s/photos/nature",
        "https://www.pexels.com/search/nature/",
        "https://pixabay.com/images/search/architecture/",
        "https://www.flickr.com/search/?text=city",
        "https://stock.adobe.com/search?k=modern+office"
    ]

    # Updated source URLs for secondary images (more product-oriented and diverse)
    secondary_sources = [
        "https://unsplash.com/s/photos/product",
        "https://www.pexels.com/search/product%20shot/",
        "https://pixabay.com/images/search/merchandise/",
        "https://www.flickr.com/search/?text=product+photography",
        "https://stock.adobe.com/search?k=modern+product",
        "https://unsplash.com/s/photos/fashion",
        "https://www.pexels.com/search/fashion/",
        "https://pixabay.com/images/search/apparel/",
        "https://www.flickr.com/search/?text=clothing",
        "https://stock.adobe.com/search?k=product+display"
    ]

    # Crawl and download images for base images.
    for url in base_sources:
        crawl_and_download(url, base_images_folder, limit=5)

    # Crawl and download images for secondary images.
    for url in secondary_sources:
        crawl_and_download(url, secondary_images_folder, limit=5)

Crawl4AI library not installed. Falling back to basic requests-based crawling.

Crawling: https://unsplash.com/s/photos/office
Falling back to basic requests for URL: https://unsplash.com/s/photos/office
Error fetching https://unsplash.com/s/photos/office: 403 Client Error: Forbidden for url: https://unsplash.com/s/photos/office
Skipping https://unsplash.com/s/photos/office due to fetch error.

Crawling: https://www.pexels.com/search/urban/
Falling back to basic requests for URL: https://www.pexels.com/search/urban/
Error fetching https://www.pexels.com/search/urban/: 403 Client Error: Forbidden for url: https://www.pexels.com/search/urban/
Skipping https://www.pexels.com/search/urban/ due to fetch error.

Crawling: https://pixabay.com/images/search/landscape/
Falling back to basic requests for URL: https://pixabay.com/images/search/landscape/
Error fetching https://pixabay.com/images/search/landscape/: 403 Client Error: Forbidden for url: https://pixabay.com/images/search/landscape/
S